In [ ]:
import json
import os
import numpy as np
import pandas as pd
import torch

from chronos import BaseChronosPipeline
from sklearn.metrics import root_mean_squared_error

import time

start_time = time.time()

# ============================================================
# CONFIG
# ============================================================
DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR  = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR   = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"

countries = ["Germany", "Ireland", "Portugal","Denmark"]
#days = ["day1", "day2", "day3", "day4", "day5"]
days = ["day1"]


features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
#    "price_eur_kwh"
]



PRED_LEN = 96
CONTEXT_TAIL = 10000
QUANTILES = [0.5]  # median

# Optional: reduce CPU threads if you want your PC responsive
# torch.set_num_threads(8)

# ============================================================
# LOAD SPLITS
# ============================================================
with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

# ============================================================
# LOAD CHRONOS-BOLT-BASE
# ============================================================
pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-bolt-base",
    device_map="cuda",  
)

print("torch:", torch.__version__)
print("pipeline model device:", pipeline.model.device)

rmse_results = []
for country in countries:
    print("\nProcessing country:", country)
    country_start_time = time.time()

    data_path = os.path.join(DATA_DIR, f"dataset_{country.capitalize()}.csv")
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    # --------------------------------------------------------
    # COUNTRY-SPECIFIC FEATURES
    # --------------------------------------------------------
    country_features = [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "precipitation",
        "direct_radiation",
    ]

    if country == "Denmark":
        country_features.append("price_eur_kwh")

    households = [c for c in df.columns if c not in country_features]

    for day in days:
        print("   Day:", day)

        cutoff = pd.to_datetime(dataset_days[country][day])

        future_index = df.loc[df.index >= cutoff].index[:PRED_LEN]

        if len(future_index) < PRED_LEN:
            inferred = pd.infer_freq(df.index)

            if inferred is None:
                deltas = df.index.to_series().diff().dropna()
                step = deltas.mode().iloc[0]

                future_index = pd.date_range(
                    start=cutoff,
                    periods=PRED_LEN,
                    freq=step
                )
            else:
                future_index = pd.date_range(
                    start=cutoff,
                    periods=PRED_LEN,
                    freq=inferred
                )

        predictions_df_all_households = pd.DataFrame(index=future_index)
        predictions_df_all_households.index.name = "timestamp"

        rmse_households = []

        for household in households:
            s_train = (
                df.loc[df.index < cutoff, household]
                .astype(float)
                .dropna()
            )

            if len(s_train) < 10:
                predictions_df_all_households[household] = np.nan
                continue

            s_train = s_train.tail(CONTEXT_TAIL)

            inputs = torch.tensor(
                s_train.values,
                dtype=torch.float32
            )

            with torch.no_grad():
                quantiles, mean = pipeline.predict_quantiles(
                    inputs=inputs,
                    prediction_length=PRED_LEN,
                    quantile_levels=QUANTILES,
                )

            if quantiles.ndim == 2:
                y_pred = quantiles[:, 0].detach().numpy()
            else:
                y_pred = quantiles[0, :, 0].detach().numpy()

            predictions_df_all_households[household] = y_pred

            y_true = df.loc[
                predictions_df_all_households.index,
                household
            ].astype(float)

            y_pred_s = predictions_df_all_households[household]

            mask = y_true.notna() & y_pred_s.notna()

            if mask.sum() > 0:
                rmse = root_mean_squared_error(
                    y_true[mask],
                    y_pred_s[mask]
                )
                rmse_households.append(rmse)

        avg_rmse = (
            float(np.mean(rmse_households))
            if rmse_households
            else np.nan
        )

        rmse_results.append({
            "country": country,
            "day": day,
            "rmse": avg_rmse
        })

        out_path = os.path.join(
            OUT_DIR,
            f"Chronos_bolt_base_Univar_pred_{day}_{country.capitalize()}.csv"
        )

        os.makedirs(os.path.dirname(out_path), exist_ok=True)

        predictions_df_all_households.to_csv(
            out_path,
            index=True
        )

        print("      Saved:", out_path)

    # ========================================================
    # COUNTRY RUNTIME
    # ========================================================
    country_runtime = time.time() - country_start_time

    print(
        f"\nTotal runtime for {country}: "
        f"{country_runtime:.2f} seconds"
    )

    # ========================================================
    # SAVE / UPDATE JSON WITHOUT OVERWRITING EXISTING CONTENT
    # ========================================================
    json_path = os.path.join(
        OUT_DIR,
        f"time_spend_{country}.json"
    )

    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            runtime_dict = json.load(f)
    else:
        runtime_dict = {}

    if "Foundational" not in runtime_dict:
        runtime_dict["Foundational"] = {}

    runtime_dict["Foundational"]["Chronosboltbase"] = country_runtime

    with open(json_path, "w") as f:
        json.dump(runtime_dict, f, indent=4)

    print(f"Saved/updated runtime JSON: {json_path}")